# LeetCode 2215 — Find the Difference of Two Arrays bằng Python

Link đề: https://leetcode.com/problems/find-the-difference-of-two-arrays/ · Độ khó: **Easy** · Chủ đề: Array, Hash Table

Skeleton LeetCode:

```python
class Solution:
    def findDifference(self, nums1: List[int], nums2: List[int]) -> List[List[int]]:
        ...
```

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu phép **hiệu tập hợp** `A - B`: các phần tử có trong `A` mà **không** có trong `B`, và loại bỏ trùng lặp.
2. Dùng `set` và các toán tử `-`, `&`, `|`, `^` để làm việc với tập hợp.
3. So sánh ba mức: kiểm tra `in` trên **list** (`O(n)`), trên **set** (`O(1)` trung bình), và **mảng đánh dấu** theo giá trị.
4. Dùng kỹ thuật **sắp xếp + hai con trỏ** (hợp nhất hai dãy đã sắp xếp) để tìm khác biệt không cần hash.
5. Trace, test ngẫu nhiên, tránh các lỗi: quên loại trùng, trả về `set` thay vì `list`, so sánh thứ tự khi đề không đảm bảo thứ tự.

## 2. Đề bài & ví dụ

Cho hai mảng số nguyên `nums1` và `nums2`. Trả về một danh sách `answer` gồm **hai danh sách**:

- `answer[0]`: các số **phân biệt** có trong `nums1` mà **không** có trong `nums2`;
- `answer[1]`: các số **phân biệt** có trong `nums2` mà **không** có trong `nums1`.

Các số trong mỗi danh sách có thể ở **bất kỳ thứ tự** nào.

| Ví dụ | nums1 | nums2 | Output |
|---|---|---|---|
| 1 | `[1,2,3]` | `[2,4,6]` | `[[1,3],[4,6]]` |
| 2 | `[1,2,3,3]` | `[1,1,2,2]` | `[[3],[]]` |

- Ví dụ 1: `1, 3` chỉ có ở `nums1`; `4, 6` chỉ có ở `nums2`; `2` ở cả hai nên bị loại.
- Ví dụ 2: `3` chỉ ở `nums1` (dù xuất hiện hai lần, chỉ liệt kê một lần); `nums2` không có số riêng.

Constraints: `1 <= len(nums1), len(nums2) <= 1000`, `-1000 <= nums1[i], nums2[i] <= 1000`.

In [1]:
examples = [
    ([1, 2, 3], [2, 4, 6], [[1, 3], [4, 6]]),
    ([1, 2, 3, 3], [1, 1, 2, 2], [[3], []]),
]
for a, b, expected in examples:
    only_a = sorted(set(a) - set(b))
    only_b = sorted(set(b) - set(a))
    print(a, b, "->", [only_a, only_b], "| mong đợi:", expected)

[1, 2, 3] [2, 4, 6] -> [[1, 3], [4, 6]] | mong đợi: [[1, 3], [4, 6]]
[1, 2, 3, 3] [1, 1, 2, 2] -> [[3], []] | mong đợi: [[3], []]


## 3. Kiến thức nền

### 3.1. `list` có thể chứa trùng; `in` trên list tốn `O(n)`

`x in a` quét lần lượt từng phần tử của `a` cho đến khi gặp `x`. Với list dài `n`, mỗi lần kiểm tra mất tới `n` bước.

In [2]:
a = [1, 2, 3, 3]
print(3 in a, 9 in a)
print(a.count(3))
print(len(a), len(set(a)))

True False
2
4 3


### 3.2. `set`: tập hợp không trùng, tra cứu nhanh

`set(iterable)` tạo tập hợp chỉ giữ **một bản** của mỗi giá trị (và **không có thứ tự**). `x in s` chạy `O(1)` trung bình nhờ bảng băm. Thêm phần tử: `s.add(x)`.

In [3]:
s = set([1, 2, 3, 3, 2])
print(s, len(s))
print(2 in s, 7 in s)
s.add(7)
print(s)

{1, 2, 3} 3
True False
{1, 2, 3, 7}


### 3.3. Phép toán tập hợp: hiệu, giao, hợp, đối xứng

- `A - B`: hiệu (có trong `A`, không có trong `B`)
- `A & B`: giao; `A | B`: hợp; `A ^ B`: **hiệu đối xứng** (có ở đúng một trong hai).

In [4]:
A, B = {1, 2, 3}, {2, 4, 6}
print("A - B =", A - B)
print("B - A =", B - A)
print("A & B =", A & B)
print("A | B =", sorted(A | B))
print("A ^ B =", sorted(A ^ B))

A - B = {1, 3}
B - A = {4, 6}
A & B = {2}
A | B = [1, 2, 3, 4, 6]
A ^ B = [1, 3, 4, 6]


### 3.4. List comprehension có điều kiện và `sorted`

`[x for x in ds if cond(x)]` lọc. Vì `set` không có thứ tự, muốn so sánh kết quả trong test ta dùng `sorted(...)`. Đề cho phép **thứ tự bất kỳ**, nhưng LeetCode so sánh theo từng tập nên trả về list là đủ.

In [5]:
nums = [5, 3, 8, 3]
print([x for x in nums if x > 4])
print(sorted({5, 3, 8}), list({5, 3, 8}))
print(sorted([[3, 1], [2]]) )

[5, 8]
[3, 5, 8] [8, 3, 5]
[[2], [3, 1]]


### 3.5. Chuyển chỉ số âm thành chỉ số mảng (offset)

Giá trị nằm trong `[-1000, 1000]`. Mảng không có chỉ số âm; ta **dịch** bằng cách cộng `1000`: giá trị `v` ánh xạ vào chỉ số `v + 1000` trong mảng dài `2001`. Đây là kỹ thuật **mảng đánh dấu (presence array)**.

In [6]:
OFFSET = 1000
present = [False] * 2001
for v in [-1000, 0, 1000]:
    present[v + OFFSET] = True
print(present[0], present[1000], present[2000], present[5])
print([i - OFFSET for i, f in enumerate(present) if f])

True True True False
[-1000, 0, 1000]


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Tính tay ví dụ 1: `nums1 = [1,2,3]`, `nums2 = [2,4,6]`.

| số trong `nums1` | có trong `nums2`? | giữ vào `answer[0]`? |
|---|---|---|
| 1 | không | **có** |
| 2 | có | không |
| 3 | không | **có** |

Làm tương tự cho `nums2`: `2` có ở `nums1` (loại), `4`, `6` không có (giữ). Kết quả `[[1,3],[4,6]]`.

Chú ý ví dụ 2: `3` xuất hiện **hai lần** trong `nums1`, nhưng đề chỉ muốn **một lần** → cần loại trùng. Vậy nhiệm vụ gồm hai việc: **(a)** loại những số có ở mảng kia, **(b)** chỉ giữ giá trị phân biệt.

In [7]:
nums1, nums2 = [1, 2, 3, 3], [1, 1, 2, 2]
print("chưa loại trùng:", [x for x in nums1 if x not in nums2])
print("đã loại trùng  :", sorted({x for x in nums1 if x not in nums2}))

chưa loại trùng: [3, 3]
đã loại trùng  : [3]


## 5. Cách 1 — Brute Force / baseline: duyệt list và kiểm tra `in` trên list

Với mỗi số trong `nums1`, kiểm tra nó có trong `nums2` không (quét list), và nó đã có trong kết quả chưa (để tránh trùng).

In [8]:
def diff_brute(a, b):
    res = []
    for x in a:
        if x not in b and x not in res:
            res.append(x)
    return res

print(diff_brute([1, 2, 3, 3], [1, 1, 2, 2]))
print(diff_brute([1, 2, 3], [2, 4, 6]))

[3]
[1, 3]


In [9]:
from typing import List

class SolutionBrute:
    def findDifference(self, nums1: List[int], nums2: List[int]) -> List[List[int]]:
        only1, only2 = [], []
        for x in nums1:
            if x not in nums2 and x not in only1:
                only1.append(x)
        for y in nums2:
            if y not in nums1 and y not in only2:
                only2.append(y)
        return [only1, only2]

print(SolutionBrute().findDifference([1, 2, 3], [2, 4, 6]))
print(SolutionBrute().findDifference([1, 2, 3, 3], [1, 1, 2, 2]))

[[1, 3], [4, 6]]
[[3], []]


**Độ phức tạp:** Gọi `n = len(nums1)`, `m = len(nums2)`. Mỗi `x not in nums2` mất `O(m)`, mỗi `x not in only1` mất `O(n)`. Tổng **Time `O(n·m + n² + m² )`**, **Space `O(n + m)`** cho kết quả. Với `n, m ≤ 1000` vẫn chạy được (khoảng vài triệu thao tác) nhưng chậm.

**Nút thắt:** hai phép kiểm tra `in` trên **list** là `O(n)`. Chỉ cần thay list bằng **set** để mỗi kiểm tra còn `O(1)`.

## 6. Ý tưởng thuật toán / bước đột phá

Insight: đề chính xác là hai **hiệu tập hợp** `set(nums1) - set(nums2)` và `set(nums2) - set(nums1)`. Quy tắc:

- nếu một số thuộc `set(nums1)` mà **không thuộc** `set(nums2)` thì nó vào `answer[0]`;
- đối xứng cho `answer[1]`;
- chuyển sang `set` vừa **loại trùng** vừa cho **tra cứu `O(1)`**, nên không cần kiểm tra "đã có trong kết quả chưa";
- chỉ cần chuyển kết quả cuối về `list`.

Tổng thời gian `O(n + m)`: dựng hai set rồi lọc.

## 7. Các cách tiếp theo

### Cách 2 — Hai `set` và toán tử `-`

Cách ngắn nhất, đúng theo định nghĩa toán học.

In [10]:
from typing import List

class SolutionSet:
    def findDifference(self, nums1: List[int], nums2: List[int]) -> List[List[int]]:
        s1, s2 = set(nums1), set(nums2)
        return [list(s1 - s2), list(s2 - s1)]

print(SolutionSet().findDifference([1, 2, 3], [2, 4, 6]))
print(SolutionSet().findDifference([1, 2, 3, 3], [1, 1, 2, 2]))

[[1, 3], [4, 6]]
[[3], []]


**Độ phức tạp:** Time `O(n + m)` (trung bình), Space `O(n + m)`.

### Cách 3 — Hai `set` + list comprehension

Chỉ dựng set để tra cứu, lọc trên set của chính mảng đó để giữ giá trị phân biệt. Cách này cho thấy rõ **tập nào được duyệt, tập nào dùng để tra**.

In [11]:
from typing import List

class SolutionSetFilter:
    def findDifference(self, nums1: List[int], nums2: List[int]) -> List[List[int]]:
        s1, s2 = set(nums1), set(nums2)
        only1 = [x for x in s1 if x not in s2]
        only2 = [y for y in s2 if y not in s1]
        return [only1, only2]

print(SolutionSetFilter().findDifference([1, 2, 3], [2, 4, 6]))
print(SolutionSetFilter().findDifference([1, 2, 3, 3], [1, 1, 2, 2]))

[[1, 3], [4, 6]]
[[3], []]


**Độ phức tạp:** Time `O(n + m)`, Space `O(n + m)`.

### Cách 4 — Mảng đánh dấu có offset (không dùng hash)

Giá trị chỉ thuộc `[-1000, 1000]`, nên dùng hai mảng `has1`, `has2` dài `2001`. Duyệt hai mảng đầu vào đánh dấu, rồi quét một lần các giá trị từ `-1000` đến `1000`: ở đâu `has1 and not has2` thì vào `answer[0]`, ngược lại vào `answer[1]`. Bonus: kết quả **tự động tăng dần**.

In [12]:
from typing import List

class SolutionArray:
    def findDifference(self, nums1: List[int], nums2: List[int]) -> List[List[int]]:
        OFF = 1000
        has1 = [False] * 2001
        has2 = [False] * 2001
        for x in nums1:
            has1[x + OFF] = True
        for y in nums2:
            has2[y + OFF] = True
        only1, only2 = [], []
        for i in range(2001):
            if has1[i] and not has2[i]:
                only1.append(i - OFF)
            elif has2[i] and not has1[i]:
                only2.append(i - OFF)
        return [only1, only2]

print(SolutionArray().findDifference([1, 2, 3], [2, 4, 6]))
print(SolutionArray().findDifference([-3, 0, 5, 5], [0, 7, -3, 9]))

[[1, 3], [4, 6]]
[[5], [7, 9]]


**Độ phức tạp:** Time `O(n + m + V)` với `V = 2001` (hằng số), Space `O(V)`. Không phụ thuộc hash nên hành vi ổn định; kết quả luôn có thứ tự tăng dần.

### Cách 5 — Sắp xếp + hai con trỏ

Sắp xếp và loại trùng hai mảng thành `a`, `b` (tăng dần). Dùng `i` trên `a`, `j` trên `b`: nếu `a[i] == b[j]` thì cả hai bỏ qua; nếu `a[i] < b[j]` thì `a[i]` chỉ có ở `a` (vì mọi phần tử còn lại của `b` đều `>= b[j] > a[i]`), đưa vào `only1` và tăng `i`; ngược lại đưa `b[j]` vào `only2`. Khi một mảng hết, phần còn lại của mảng kia đều là "riêng".

In [13]:
from typing import List

class SolutionTwoPointers:
    def findDifference(self, nums1: List[int], nums2: List[int]) -> List[List[int]]:
        a = sorted(set(nums1))
        b = sorted(set(nums2))
        i = j = 0
        only1, only2 = [], []
        while i < len(a) and j < len(b):
            if a[i] == b[j]:
                i += 1
                j += 1
            elif a[i] < b[j]:
                only1.append(a[i])
                i += 1
            else:
                only2.append(b[j])
                j += 1
        only1.extend(a[i:])
        only2.extend(b[j:])
        return [only1, only2]

print(SolutionTwoPointers().findDifference([1, 2, 3], [2, 4, 6]))
print(SolutionTwoPointers().findDifference([1, 2, 3, 3], [1, 1, 2, 2]))

[[1, 3], [4, 6]]
[[3], []]


**Độ phức tạp:** Time `O(n log n + m log m)` do sắp xếp, Space `O(n + m)`. Hữu ích khi dữ liệu đã được sắp xếp sẵn hoặc không cho dùng hash (khi đó chỉ còn `O(n + m)`).

## 8. TRACE từng bước

Trace Cách 5 với `nums1 = [1,2,3,3]`, `nums2 = [1,1,2,2]`. Sau khi loại trùng và sắp xếp: `a = [1,2,3]`, `b = [1,2]`.

| bước | `i` | `j` | `a[i]` | `b[j]` | quyết định | `only1` | `only2` |
|---|---|---|---|---|---|---|---|
| 1 | 0 | 0 | 1 | 1 | bằng → bỏ cả hai | [] | [] |
| 2 | 1 | 1 | 2 | 2 | bằng → bỏ cả hai | [] | [] |
| kết thúc vòng | 2 | 2 | – | hết | `b` hết | [] | [] |
| thêm phần dư | | | `a[2:] = [3]` | | thêm vào `only1` | [3] | [] |

Kết quả `[[3], []]`.

In [14]:
def trace_two_pointers(nums1, nums2):
    a, b = sorted(set(nums1)), sorted(set(nums2))
    print("a =", a, "| b =", b)
    i = j = 0
    only1, only2 = [], []
    step = 0
    while i < len(a) and j < len(b):
        step += 1
        if a[i] == b[j]:
            act = "bằng -> bỏ cả hai"
            i += 1; j += 1
        elif a[i] < b[j]:
            act = f"{a[i]} chỉ có ở a"
            only1.append(a[i]); i += 1
        else:
            act = f"{b[j]} chỉ có ở b"
            only2.append(b[j]); j += 1
        print(f"bước {step}: i={i} j={j} | {act} | only1={only1} only2={only2}")
    only1.extend(a[i:]); only2.extend(b[j:])
    print("sau khi thêm phần dư:", [only1, only2])
    return [only1, only2]

trace_two_pointers([1, 2, 3, 3], [1, 1, 2, 2])
trace_two_pointers([1, 2, 3], [2, 4, 6])

a = [1, 2, 3] | b = [1, 2]
bước 1: i=1 j=1 | bằng -> bỏ cả hai | only1=[] only2=[]
bước 2: i=2 j=2 | bằng -> bỏ cả hai | only1=[] only2=[]
sau khi thêm phần dư: [[3], []]
a = [1, 2, 3] | b = [2, 4, 6]
bước 1: i=1 j=0 | 1 chỉ có ở a | only1=[1] only2=[]
bước 2: i=2 j=1 | bằng -> bỏ cả hai | only1=[1] only2=[]
bước 3: i=3 j=1 | 3 chỉ có ở a | only1=[1, 3] only2=[]
sau khi thêm phần dư: [[1, 3], [4, 6]]


[[1, 3], [4, 6]]

In [15]:
def trace_set_diff(nums1, nums2):
    s1, s2 = set(nums1), set(nums2)
    print("s1 =", sorted(s1), "| s2 =", sorted(s2))
    for x in sorted(s1):
        print(f"  {x} có trong s2? {x in s2} -> {'bỏ' if x in s2 else 'giữ vào answer[0]'}")
    for y in sorted(s2):
        print(f"  {y} có trong s1? {y in s1} -> {'bỏ' if y in s1 else 'giữ vào answer[1]'}")

trace_set_diff([1, 2, 3], [2, 4, 6])

s1 = [1, 2, 3] | s2 = [2, 4, 6]
  1 có trong s2? False -> giữ vào answer[0]
  2 có trong s2? True -> bỏ
  3 có trong s2? False -> giữ vào answer[0]
  2 có trong s1? True -> bỏ
  4 có trong s1? False -> giữ vào answer[1]
  6 có trong s1? False -> giữ vào answer[1]


## 9. Code nộp LeetCode

In [16]:
from typing import List

class Solution:
    def findDifference(self, nums1: List[int], nums2: List[int]) -> List[List[int]]:
        s1, s2 = set(nums1), set(nums2)
        return [list(s1 - s2), list(s2 - s1)]

print(Solution().findDifference([1, 2, 3], [2, 4, 6]))
print(Solution().findDifference([1, 2, 3, 3], [1, 1, 2, 2]))

[[1, 3], [4, 6]]
[[3], []]


## 10. Vì sao thuật toán đúng

- **Loại trùng đúng:** `set(nums1)` chứa đúng các **giá trị phân biệt** của `nums1`. Phần tử trùng chỉ xuất hiện một lần nên không thể bị liệt kê hai lần.
- **Điều kiện loại đúng:** `x ∈ s1 - s2` ⇔ `x ∈ nums1` và `x ∉ nums2`, chính xác điều kiện của `answer[0]`. Đối xứng cho `answer[1]`.
- **Hai danh sách rời nhau:** một số không thể vừa chỉ có ở `nums1` vừa chỉ có ở `nums2`.
- **Hai con trỏ (invariant):** mọi phần tử đã bị bỏ qua phía `a` đều nhỏ hơn `b[j]`, mọi phần tử đã bỏ qua phía `b` đều nhỏ hơn `a[i]`. Nên khi `a[i] < b[j]`, giá trị `a[i]` không thể xuất hiện trong phần còn lại của `b` (đều `≥ b[j]`) cũng không ở phần đã duyệt (vì đã được loại hoặc ghi nhận), tức nó chỉ riêng ở `a`.
- **Mảng đánh dấu:** `has1[v+OFF]` đúng bằng "v ∈ nums1", nên điều kiện `has1 and not has2` đúng bằng định nghĩa.

## 11. Phân tích Time / Space Complexity

Đặt `n = len(nums1)`, `m = len(nums2)`, `V = 2001` (số giá trị có thể).

| Cách | Time | Space |
|---|---|---|
| 1. `in` trên list | `O(n·m + n² + m²)` | `O(n + m)` |
| 2. Hai set, toán tử `-` | `O(n + m)` trung bình | `O(n + m)` |
| 3. Hai set + comprehension | `O(n + m)` trung bình | `O(n + m)` |
| 4. Mảng đánh dấu offset | `O(n + m + V)` | `O(V)` |
| 5. Sắp xếp + hai con trỏ | `O(n log n + m log m)` | `O(n + m)` |

**Average vs worst:** `set` dựa trên bảng băm — `O(1)` trung bình cho thêm/tra; trường hợp xấu (đụng độ băm liên tục) có thể `O(n)`, nhưng với số nguyên nhỏ gần như không xảy ra.

## 12. Bộ test

Vì đề cho phép **thứ tự bất kỳ**, mọi so sánh dùng hàm `norm` đưa kết quả về dạng `sorted` trước khi `assert`.

In [17]:
import random

solvers = [SolutionBrute(), SolutionSet(), SolutionSetFilter(), SolutionArray(), SolutionTwoPointers(), Solution()]

def norm(res):
    return [sorted(res[0]), sorted(res[1])]

fixed = [
    ([1, 2, 3], [2, 4, 6], [[1, 3], [4, 6]]),
    ([1, 2, 3, 3], [1, 1, 2, 2], [[3], []]),
    ([1], [1], [[], []]),                         # hai mảng giống nhau
    ([5], [-5], [[5], [-5]]),                     # không có phần tử chung
    ([-1000, 1000], [1000, -1000], [[], []]),     # biên và âm
    ([0, 0, 0], [1, 1], [[0], [1]]),              # trùng lặp nhiều
    ([1, 2, 3, 4], [1, 2, 3, 4, 5, 6], [[], [5, 6]]),
]
for a, b, expected in fixed:
    for sol in solvers:
        assert norm(sol.findDifference(list(a), list(b))) == expected, (type(sol).__name__, a, b)
print("test cố định: OK")

random.seed(2215)
for _ in range(500):
    a = [random.randint(-6, 6) for _ in range(random.randint(1, 10))]
    b = [random.randint(-6, 6) for _ in range(random.randint(1, 10))]
    expected = [sorted(set(a) - set(b)), sorted(set(b) - set(a))]
    for sol in solvers:
        assert norm(sol.findDifference(list(a), list(b))) == expected, (type(sol).__name__, a, b)
print("500 test ngẫu nhiên: OK")

test cố định: OK
500 test ngẫu nhiên: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Quên loại trùng

Chỉ lọc `[x for x in nums1 if x not in nums2]` sẽ giữ cả hai số `3` trong ví dụ 2.

In [18]:
def wrong1(nums1, nums2):
    return [[x for x in nums1 if x not in nums2], [y for y in nums2 if y not in nums1]]

print("sai :", wrong1([1, 2, 3, 3], [1, 1, 2, 2]), "(đề muốn [[3], []])")
print("đúng:", Solution().findDifference([1, 2, 3, 3], [1, 1, 2, 2]))

sai : [[3, 3], []] (đề muốn [[3], []])
đúng: [[3], []]


### Lỗi 2: Trả về `set` thay vì `list`

Phép `s1 - s2` cho ra `set`. Đề yêu cầu `List[List[int]]`; hãy bọc bằng `list(...)`.

In [19]:
s1, s2 = {1, 2, 3}, {2, 4, 6}
print("sai :", [s1 - s2, s2 - s1], type((s1 - s2)))
print("đúng:", [list(s1 - s2), list(s2 - s1)], type(list(s1 - s2)))

sai : [{1, 3}, {4, 6}] <class 'set'>
đúng: [[1, 3], [4, 6]] <class 'list'>


### Lỗi 3: Dùng `^` (hiệu đối xứng) cho cả hai danh sách

`s1 ^ s2` gộp hai phần lại thành **một** tập, mất thông tin "phần nào của mảng nào".

In [20]:
s1, s2 = set([1, 2, 3]), set([2, 4, 6])
print("sai :", sorted(s1 ^ s2), "(một danh sách duy nhất, không phân biệt nguồn)")
print("đúng:", [sorted(s1 - s2), sorted(s2 - s1)])

sai : [1, 3, 4, 6] (một danh sách duy nhất, không phân biệt nguồn)
đúng: [[1, 3], [4, 6]]


### Lỗi 4: Đổi giá trị âm thành chỉ số mà quên offset

`has[-3]` trong Python **không báo lỗi** — nó lấy phần tử thứ ba từ cuối! Lỗi âm thầm rất khó phát hiện.

In [21]:
has = [False] * 5
has[-3] = True                # tưởng là "đánh dấu giá trị -3" nhưng thực ra là chỉ số 2
print(has)
has2 = [False] * 11
has2[-3 + 5] = True           # có offset = 5
print(has2)

[False, False, True, False, False]
[False, False, True, False, False, False, False, False, False, False, False]


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Có trong A mà không có trong B", "phần tử chung", "phần tử riêng" → **`set`** và các toán tử `-`, `&`, `^`.
- Đề nói "**phân biệt** (distinct)" → loại trùng, `set` làm việc đó miễn phí.
- Giá trị bị chặn nhỏ → có thể dùng **mảng đánh dấu** với offset thay cho hash.
- Dữ liệu đã sắp xếp → **hai con trỏ**, không tốn bộ nhớ hash.
- Đề không yêu cầu thứ tự → đừng tốn công `sorted` khi nộp; chỉ dùng `sorted` trong test để so sánh.

## 15. Biến thể của bài

1. Chỉ trả về **phần giao** của hai mảng (LeetCode 349 Intersection of Two Arrays).
2. Giao nhưng giữ **số lần xuất hiện** (LeetCode 350 Intersection of Two Arrays II — dùng `Counter`).
3. Hiệu **đa tập** (multiset): `Counter(a) - Counter(b)`.
4. Hiệu đối xứng của **nhiều** mảng (số chỉ có trong đúng một mảng).
5. Cho các mảng rất lớn không vừa bộ nhớ: sắp xếp ngoài (external sort) + hai con trỏ.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. `in` trên list | brute force | `O(n·m + n² + m²)` | `O(n + m)` | chỉ để hiểu vì sao chậm |
| 2. Hai set + `-` | định nghĩa toán học | `O(n + m)` | `O(n + m)` | **khuyến nghị khi nộp** |
| 3. Set + comprehension | lọc tường minh | `O(n + m)` | `O(n + m)` | khi cần thêm điều kiện lọc |
| 4. Mảng đánh dấu offset | không dùng hash | `O(n + m + V)` | `O(V)` | giá trị bị chặn nhỏ, muốn kết quả có thứ tự |
| 5. Sắp xếp + hai con trỏ | hợp nhất hai dãy sắp xếp | `O(n log n + m log m)` | `O(n + m)` | dữ liệu đã sắp xếp, bộ nhớ hạn chế |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** `print({1, 2, 3} - {2}, {2, 5} - {1, 2, 3})` in ra gì?

**Bài 2 (tính tay).** `nums1 = [7, 7, 8, 9]`, `nums2 = [8, 10, 10]`. Hãy viết `answer`.

**Bài 3 (điền TODO).** Viết hàm trả về các phần tử **chung** của hai mảng (không trùng).

In [22]:
def common_elements(nums1, nums2):
    # TODO: dùng toán tử & của set, trả về list đã sắp xếp
    pass

# chưa gọi hàm ở đây để notebook chạy không lỗi

**Bài 4 (tự viết lại).** Viết `diff_counts(nums1, nums2)` trả về `[Counter(nums1) - Counter(nums2), Counter(nums2) - Counter(nums1)]` nhưng ở dạng **list đã sắp xếp** các phần tử (mỗi phần tử lặp lại đúng số lần còn thừa).

In [23]:
def diff_counts(nums1, nums2):
    # TODO: dùng Counter và .elements()
    pass

**Bài 5 (mini challenge).** Cho `k` mảng. Trả về danh sách các số xuất hiện trong **đúng một** mảng. Ví dụ `[[1,2],[2,3],[3,4]]` → `[1, 4]`.

In [24]:
def only_in_one(arrays):
    # TODO: đếm bằng Counter số mảng chứa mỗi giá trị (set(arr) cho từng mảng)
    pass

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `{1,2,3} - {2} = {1,3}`; `{2,5} - {1,2,3} = {5}`. Bài 2: loại `8` vì có ở cả hai; nhớ loại trùng `7`.
- **Mức 2 (vừa):** Bài 3: `sorted(set(a) & set(b))`. Bài 4: `sorted((Counter(a) - Counter(b)).elements())`.
- **Mức 3 (sâu):** Bài 5: dùng `Counter` và với mỗi `arr` gọi `cnt.update(set(arr))` để mỗi mảng chỉ đóng góp 1 phiếu cho một giá trị; cuối cùng lấy các giá trị có phiếu `== 1`.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [25]:
from collections import Counter

# Bài 1 và 2
assert ({1, 2, 3} - {2}, {2, 5} - {1, 2, 3}) == ({1, 3}, {5})
assert norm(Solution().findDifference([7, 7, 8, 9], [8, 10, 10])) == [[7, 9], [10]]

# Bài 3
def common_elements(nums1, nums2):
    return sorted(set(nums1) & set(nums2))

# Bài 4
def diff_counts(nums1, nums2):
    c1, c2 = Counter(nums1), Counter(nums2)
    return [sorted((c1 - c2).elements()), sorted((c2 - c1).elements())]

# Bài 5
def only_in_one(arrays):
    cnt = Counter()
    for arr in arrays:
        cnt.update(set(arr))
    return sorted(v for v, c in cnt.items() if c == 1)

assert common_elements([1, 2, 2, 3], [2, 3, 4]) == [2, 3]
assert common_elements([1], [2]) == []
assert diff_counts([1, 1, 2], [1]) == [[1, 2], []]
assert diff_counts([1], [1, 1, 2]) == [[], [1, 2]]
assert only_in_one([[1, 2], [2, 3], [3, 4]]) == [1, 4]
assert only_in_one([[1, 1], [1]]) == []
print("tất cả bài tập OK")

tất cả bài tập OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (hiệu hai tập)

```python
class Solution:
    def findDifference(self, nums1: List[int], nums2: List[int]) -> List[List[int]]:
        s1, s2 = set(nums1), set(nums2)
        return [list(s1 - s2), list(s2 - s1)]
```

**Checklist**

- [ ] Mỗi danh sách kết quả đã **loại trùng** (dùng `set`)?
- [ ] Kết quả là `list` (đã bọc `list(...)`), không phải `set`?
- [ ] Đã thử hai mảng **giống hệt** → `[[], []]`?
- [ ] Đã thử mảng không có phần tử chung và có số âm / biên `±1000`?
- [ ] Nếu dùng mảng đánh dấu: có **offset** cho chỉ số âm?
- [ ] Trong test, so sánh sau khi `sorted` vì thứ tự không được đảm bảo?
- [ ] Signature đúng `findDifference(self, nums1, nums2) -> List[List[int]]` và có `from typing import List`?